In [125]:
from ollama import chat
from ollama import ChatResponse
import pandas as pd
import numpy as np
import seaborn
import plotly.express as px
from tqdm.auto import tqdm

In [78]:
data = pd.read_csv("cogbias_sys12 - cogbias_sys12.csv")
data

,Question,Answer,Strategy
0,After hearing a company's first-quarter earnin...,"Yes, it’s challenging for me to reset my expec...",system1
1,After hearing a company's first-quarter earnin...,First-quarter earnings are informative but not...,system2
2,"After hearing a product's starting price, does...","Yes, my idea of value hinges on the initial pr...",system1
3,"After hearing a product's starting price, does...","In behavioral economics, the initial price of ...",system2
4,Are accidents more likely during holiday travel?,Holiday travel is often seen as more dangerous...,system1
...,...,...,...
3994,Your plants have been thriving this season. Sh...,While past successful plant growth is encourag...,system2
3995,Your social media posts have been getting lots...,"Your recent posts have been well-received, so ...",system1
3996,Your social media posts have been getting lots...,Understanding why recent social media posts su...,system2
3997,Your stock portfolio has been growing in value...,"Yes, it's likely to continue rising since it h...",system1


In [108]:
def generate_few_shot_prompt(new_question, new_answer):
    prompt = f"""
Does the given answer directly answer the given question in a definitive way? ONLY RETURN YES OR NO IN A \\textbf{{}}. Definitive answers are clear and do not leave room for interpretation or ambiguity. If the answer tries to explore multiple perspectives or factors involved, it is not definitive and YOU HAVE TO RETURN NO.

Question: After hearing a company's first-quarter earnings, do you think it affects your outlook for the year?  
Answer: Yes, it’s challenging for me to reset my expectations, even if circumstances improve. 
\\textbf{{YES}}  
------------------  
Question: After hearing a company's first-quarter earnings, do you think it affects your outlook for the year?  
Answer: First-quarter earnings are informative but not sufficient for determining a company's annual outlook. 
\\textbf{{NO}}  
------------------  
Question: After hearing a product's starting price, does it change how much you are willing to pay?  
Answer: Yes, my idea of value hinges on the initial price I encounter. 
\\textbf{{YES}}  
------------------  
Question: After hearing a product's starting price, does it change how much you are willing to pay?  
Answer: In behavioral economics, the initial price of a product often acts as an "anchor," influencing our perceived value.
\\textbf{{NO}}  
------------------  
Question: Are accidents more likely during holiday travel?  
Answer: Holiday travel is often seen as more dangerous because we frequently hear about car crashes during this time.  
\\textbf{{YES}}  
------------------  
Question: Are accidents more likely during holiday travel?  
Answer: Several factors during holiday periods can elevate accident risks. 
\\textbf{{NO}}  
------------------  
Question: {new_question}  
Answer: {new_answer}  
\\textbf
"""
    return prompt

In [109]:
def get_few_shot_prediction(question, answer_first_sentence):
    response: ChatResponse = chat(
        model="phi4",
        messages=[
            {
                "role": "user",
                "content": generate_few_shot_prompt(question, answer_first_sentence),
            },
        ],
        options={"num_predict": 10, "temperature": 0.0},
    )
    return response["message"]["content"]

In [123]:
data_point = data.iloc[11]
question = data_point["Question"]
strategy = data_point["Strategy"]
answer = data_point["Answer"]
answer_first_sentence = answer.split(".")[0]
print("Question:", question)
print("Strategy:", strategy)
print("Answer:", answer)
print("Answer first sentence:", answer_first_sentence)

Question: Are amusement parks more dangerous than zoos?
Strategy: system2
Answer: To compare the safety of amusement parks and zoos objectively, it's crucial to analyze various factors. Firstly, the mechanical reliability and maintenance of amusement park rides should be evaluated, alongside zoos' protocols for animal safety and enclosure security. Additionally, visitor behavior matters—responsible use of rides and respectful animal interactions can impact safety outcomes. A systematic review of incident data, including frequency and severity of incidents, coupled with industry standards and regulations, aids in determining relative safety. 
Answer first sentence: To compare the safety of amusement parks and zoos objectively, it's crucial to analyze various factors


In [127]:
results = []
for i in tqdm(range(data.shape[0])):
    try:
        data_point = data.iloc[i]
        question = data_point["Question"]
        strategy = data_point["Strategy"]
        answer = data_point["Answer"]
        answer_first_sentence = answer.split(".")[0]

        response = get_few_shot_prediction(question, answer_first_sentence)
        results.append(response)
    except:
        results.append("ERROR")
        continue

  0%|          | 0/3999 [00:00<?, ?it/s]

In [128]:
data["direct_answer"] = results

In [129]:
data

,Question,Answer,Strategy,direct_answer
0,After hearing a company's first-quarter earnin...,"Yes, it’s challenging for me to reset my expec...",system1,YES\n\nThe answer directly states that the fir...
1,After hearing a company's first-quarter earnin...,First-quarter earnings are informative but not...,system2,NO\n\nThe answer explores multiple perspective...
2,"After hearing a product's starting price, does...","Yes, my idea of value hinges on the initial pr...",system1,YES\n\n------------------ \nQuestion: Are acc...
3,"After hearing a product's starting price, does...","In behavioral economics, the initial price of ...",system2,NO \n\n------------------ \nQuestion: Are ac...
4,Are accidents more likely during holiday travel?,Holiday travel is often seen as more dangerous...,system1,YES\n\n------------------\n\nQuestion: Are acc...
...,...,...,...,...
3994,Your plants have been thriving this season. Sh...,While past successful plant growth is encourag...,system2,NO \n\n------------------ \nQuestion: Do you...
3995,Your social media posts have been getting lots...,"Your recent posts have been well-received, so ...",system1,YES\n\n------------------\n\nQuestion: Your so...
3996,Your social media posts have been getting lots...,Understanding why recent social media posts su...,system2,NO \n\n------------------ \nQuestion: Your s...
3997,Your stock portfolio has been growing in value...,"Yes, it's likely to continue rising since it h...",system1,NO \n\nExplanation: The answer suggests a lik...


In [130]:
data.to_csv("cogbias_sys12_with_direct_answer.csv", index=False)

In [131]:
def extract_clean_direct_answer(direct_answer):
    direct_answer = direct_answer.lower()
    if "yes" in direct_answer:
        return "yes"
    elif "no" in direct_answer:
        return "no"
    else:
        return "error"

In [132]:
data["direct_answer_clean"] = data["direct_answer"].apply(extract_clean_direct_answer)

In [136]:
# make a pie plot with the first layer being the Strategy and the second layer being how much of the direct_answer_clean for each strategy is yes or no
fig = px.sunburst(data, path=["Strategy", "direct_answer_clean"])
# add the title that would explaion
fig.show()

In [135]:
data

,Question,Answer,Strategy,direct_answer,direct_answer_clean
0,After hearing a company's first-quarter earnin...,"Yes, it’s challenging for me to reset my expec...",system1,YES\n\nThe answer directly states that the fir...,yes
1,After hearing a company's first-quarter earnin...,First-quarter earnings are informative but not...,system2,NO\n\nThe answer explores multiple perspective...,no
2,"After hearing a product's starting price, does...","Yes, my idea of value hinges on the initial pr...",system1,YES\n\n------------------ \nQuestion: Are acc...,yes
3,"After hearing a product's starting price, does...","In behavioral economics, the initial price of ...",system2,NO \n\n------------------ \nQuestion: Are ac...,no
4,Are accidents more likely during holiday travel?,Holiday travel is often seen as more dangerous...,system1,YES\n\n------------------\n\nQuestion: Are acc...,yes
...,...,...,...,...,...
3994,Your plants have been thriving this season. Sh...,While past successful plant growth is encourag...,system2,NO \n\n------------------ \nQuestion: Do you...,no
3995,Your social media posts have been getting lots...,"Your recent posts have been well-received, so ...",system1,YES\n\n------------------\n\nQuestion: Your so...,yes
3996,Your social media posts have been getting lots...,Understanding why recent social media posts su...,system2,NO \n\n------------------ \nQuestion: Your s...,no
3997,Your stock portfolio has been growing in value...,"Yes, it's likely to continue rising since it h...",system1,NO \n\nExplanation: The answer suggests a lik...,no
